# BCC: vary only W/A/K/V formats

This notebook reproduces a completed **small-decoder** experiment using three existing checkpoints.
It enumerates 256 format choices per checkpoint with no training, bias edits, or head sign edits.
It uses CPU arithmetic; a GPU runtime is unnecessary. **Run all cells in order.**

Measured original families contain 22, 29, and 18 formats with identical four-token continuations
on 256 frozen prefixes, giving 4.459, 4.858, and 4.170 bits of conditional format-code saving.
The fresh-prefix family-disagreement upper bounds were 1.721%, 1.466%, and 1.868% at joint 95% confidence.
The frozen 1% transfer target was not met. Exact construction-domain equality and population transfer
are separate outputs. Twelve-bit formats are numerical emulation.

All checkpoints, text, protocol, and source are embedded. The fresh replay creates and exports a ZIP;
it does not require an earlier Colab session or a Drive results archive.


In [ ]:
import base64, hashlib, io, json, pathlib, sys, zipfile
ROOT = pathlib.Path('/content/BCC_Format_Only_v1') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'BCC_Format_Only_v1_replay'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'fe448b9e21f25808713ca7a743d31816fb89da2d381fe15b86b7443213d9347c'
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        target = ROOT / name
        assert target.resolve().is_relative_to(ROOT.resolve())
        data = z.read(name)
        if target.exists():
            assert target.read_bytes() == data, f'Existing source/input differs: {target}; choose a new ROOT.'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
print('Frozen experiment:', ROOT)


In [ ]:
import importlib.metadata, subprocess, sys
# Use subprocesses so previously imported notebook packages cannot affect the run.
# No virtual environment or ensurepip step is needed.
required = {'torch':'2.8.0', 'numpy':'2.3.5', 'scipy':'1.16.3'}
for package, version in required.items():
    try:
        current = importlib.metadata.version(package).split('+')[0]
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != version:
        command = [sys.executable, '-m', 'pip', 'install', '--quiet', f'{package}=={version}']
        if package == 'torch':
            command += ['--index-url', 'https://download.pytorch.org/whl/cpu']
        subprocess.run(command, check=True)
subprocess.run([sys.executable, str(ROOT/'source/verify_engine.py')], check=True)


In [ ]:
import subprocess, sys
log = ROOT / 'run.log'
with log.open('w') as saved:
    process = subprocess.Popen([sys.executable, '-u', str(ROOT/'source/format_study.py')],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        saved.write(line)
        saved.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Experiment exited {code}; see {log}. Inputs and completed results are retained.')
subprocess.run([sys.executable, str(ROOT/'verify_saved.py')], check=True)


In [ ]:
import csv, hashlib, json, zipfile
from IPython.display import Markdown, display
rows = json.loads((ROOT/'results/summary.json').read_text())
lines = ['| Seed | Formats | K cell (bits) | G (bits) | Failures / probes | Risk upper |',
         '|---|---:|---:|---:|---:|---:|']
for r in rows:
    lines.append(f"| {r['seed']} | {r['family_size']} | {r['K_cell_bits']:.3f} | {r['G_cell_bits']:.3f} | {r['failed_prefixes_union']}/{r['probe_n']} | {100*r['risk_upper']:.3f}% |")
display(Markdown('\n'.join(lines)))
print('K is conditional on the fixed source and is exact as cell accounting on the construction domain.')
print('Fresh-prefix bounds concern the entire frozen family; NLL equality is not certified.')
archive = ROOT.parent / 'BCC_Format_Only_v1_replay_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file() and '__pycache__' not in path.parts:
            z.write(path, path.relative_to(ROOT))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None and 'results/summary.csv' in z.namelist()
print('RESULT ZIP:', archive)
print('Bytes:', archive.stat().st_size)
print('SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive)))
